# Spatial Analysis of Multiplex Immunofluorescence Data

This notebook demonstrates how to perform spatial analysis on multiplex immunofluorescence data. The focus is on exploring the distribution and relationship of different cell populations within regions of interest (ROIs), using several advanced analysis techniques, including:

- **Mask creation**: Generating CK, NGFR, and DAPI masks from OME-TIFF images.
- **Subpopulation characterization**: Analyzing cell populations based on specific markers.
- **Distance analysis**: Computing the distances between different cell populations and specific markers.
- **Visualization**: Generating various plots, including composite images, cell-count heatmaps, and distance-based boxplots.
- **Subpopulation comparison**: Comparing cell populations across different conditions and ROIs using statistical and visual analysis.

Throughout this notebook, we will utilize various tools from the `multiplex_pipeline` package to process and analyze the data, enabling insights into the spatial organization of immune cell infiltration in tissue samples.

### Table of Contents
1. **Setup and Imports**
2. **Load OME-TIFF Images**
3. **Load DAPI Masks and Plot**
4. **Create CK Masks**
5. **Create NGFR Masks**
6. **Compute Intensity Metrics**
7. **Convert Intensities to Binary**
8. **Plot Cell-Count Combinations for Tumor Characterization**
9. **Plot Cell-Count Combinations for Tumor Infiltration**
10. **Plot Cell-Count Combinations for NGFR-related Infiltration**
11. **Overlay Conditional Cell Plots**
12. **Compute and Analyze Mask Area Summaries**
13. **Compute and Analyze Cell Densities**
14. **Compute Distances Between Subpopulations and Masks**
15. **Visualize with Boxplots and Heatmaps**

The notebook follows a structured flow that ensures each step in the analysis is reproducible and easy to follow.


### 1. Setup and Imports

In [ ]:
import os

# Set data paths if not using environment variables
# os.environ.setdefault("MULTIPLEX_DATA_DIR", "/path/to/your/data")
# os.environ.setdefault("MULTIPLEX_RESULTS_DIR", "results_spatial_analysis")

In [ ]:
import sys
from pathlib import Path

# Locate the repo root from this notebook's working directory
_repo = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
if not (_repo / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from notebooks/ or the repo root.")

# Add repo parent to sys.path so 'multiplex_pipeline' (editable-install symlink) is found
if str(_repo.parent) not in sys.path:
    sys.path.insert(0, str(_repo.parent))

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from multiplex_pipeline.io.loaders import load_csv_data, load_ome_tif_images, load_dapi_masks
from multiplex_pipeline.utils.helpers import invert_dict
from multiplex_pipeline.preprocessing.segmentation import create_channel_masks
from multiplex_pipeline.analysis.intensity import process_roi, intensity_to_binary
from multiplex_pipeline.analysis.spatial import (
    compute_mask_area_summary,
    compute_subpop_cells_per_area,
)
from multiplex_pipeline.visualization.overlays import (
    plot_conditional_cells_channels,
    compute_and_plot_subpop_distances_for_all_rois,
    compute_and_save,
)
from multiplex_pipeline.visualization.qc import (
    plot_masks,
    plot_combination_counts,
    generate_boxplots_nested,
    generate_combined_boxplots,
)
from multiplex_pipeline.schema import DISTANCE_CK_POSITIVE, DISTANCE_CK_NEGATIVE, ROI
from multiplex_pipeline.config import (
    DATA_FOLDER,
    EXPORT_DAPI_FOLDER,
    CK_SETTINGS,
    NGFR_SETTINGS,
    PIXEL_AREA,
    PIXEL_SIZE,
    CELL_COUNT_OUTPUT_DIR,
    CELL_DENSITY_OUTPUT_DIR,
    DISTANCES_SUBPOP_DIR,
    DISTANCES_POPULATIONS_DIR,
    BOXPLOTS_DISTANCES_DIR,
    BOXPLOTS_DISTANCES_HEATMAPS_DIR,
    DEFAULT_BRIGHTNESS,
    ROIS_TO_ANALYZE,
    SHADING_COLORS,
)
from multiplex_pipeline.domain import (
    CHANNELS_OF_INTEREST,
    MARKER_LABELS,
    INTENSITY_THRESHOLDS,
    CHARACTERIZATION_COMBINATIONS,
    INFILTRATION_COMBINATIONS,
    NGFR_INFILTRATION_COMBINATIONS,
    SUBPOPULATIONS,
    SUBPOPULATION_A_POSITIVE,
    SUBPOPULATION_A_NEGATIVE,
    CONDITION_COLUMN_MAP,
)

### 2. Load OME-TIFF Images

In [ ]:
images_dict = load_ome_tif_images(DATA_FOLDER)

# Verify the loaded images
print(f"{len(images_dict)} images loaded.")
for name, image in images_dict.items():
    print(f"{name}: {image.shape}")


### 3. Load DAPI Masks and Plot

In [ ]:
dapi_masks_dict = load_dapi_masks(EXPORT_DAPI_FOLDER)
plot_masks(dapi_masks_dict)


### 4. Create CK Masks

Pan-Cytokeratin (CK) marks tumor epithelium. Thresholding is done at `T = μ_ROI + k·σ_ROI`
with parameters from `CK_SETTINGS` in `config.py`.

In [ ]:
ck_masks_dict = create_channel_masks(images_dict, dapi_masks_dict, CK_SETTINGS)

# Verify the CK masks
print("Keys in ck_masks_dict:", list(ck_masks_dict.keys()))


### 5. Create NGFR Masks

NGFR defines a CK⁺ subpopulation with distinct proliferation and immunoregulatory characteristics.
Per-ROI threshold scores are set in `NGFR_SETTINGS` in `config.py`.

In [ ]:
ngfr_masks_dict = create_channel_masks(images_dict, dapi_masks_dict, NGFR_SETTINGS)

# Verify the NGFR masks
print("Keys in ngfr_masks_dict:", list(ngfr_masks_dict.keys()))


### 6. Extract Per-Cell Intensities

Processes each ROI in parallel. Returns a DataFrame with one row per cell: area, centroid,
mean fluorescence per channel, and binary mask-based flags for CK and NGFR.

In [ ]:
roi_names = [
    name for name in images_dict
    if any(roi.lower() in name.lower() for roi in ROIS_TO_ANALYZE)
]

results = [
    process_roi(
        name, images_dict[name],
        dapi_masks_dict, ck_masks_dict, ngfr_masks_dict,
        CHANNELS_OF_INTEREST, MARKER_LABELS, PIXEL_AREA,
    )
    for name in roi_names
]

df_results = pd.concat([r for r in results if r is not None], ignore_index=True)
print(f"{len(df_results)} cells extracted from {len(roi_names)} ROIs.")
df_results.head()

### 7. Apply Adaptive Thresholds → Binary Positivity

Each marker column is binarized using `T = μ_ROI + k·σ_ROI`.
Thresholds `k` are defined per marker in `INTENSITY_THRESHOLDS` in `domain.py`.

In [ ]:
df_binary = intensity_to_binary(df_results, INTENSITY_THRESHOLDS)
df_binary.head()


### 8. Tumor Characterization Cell Counts

CK⁺ cells stratified by NGFR± and functional markers (Ki-67, PD-L1, IFN-γ, HLA-DR).

In [ ]:
counts_df_tumor = plot_combination_counts(
    df=df_binary,
    rois=ROIS_TO_ANALYZE,
    combinations=CHARACTERIZATION_COMBINATIONS,
    output_dir=CELL_COUNT_OUTPUT_DIR,
    base_filename="Tumor_Characterization",
    plot_title="Tumor Characterization: Counts by Combination and ROI",
)
counts_df_tumor

### 9. Immune Infiltration Counts

Intratumoral (CK⁺) vs. stromal (CK⁻) distribution of immune subpopulations (Tregs, CD8⁺ T cells, macrophages, DCs).

In [ ]:
counts_df_tumor = plot_combination_counts(
    df=df_binary,
    rois=ROIS_TO_ANALYZE,
    combinations=INFILTRATION_COMBINATIONS,
    output_dir=CELL_COUNT_OUTPUT_DIR,
    base_filename="Tumor_Infiltration",
    plot_title="Tumor Infiltration (Immune Cells): Counts by Combination and ROI"
)
counts_df_tumor


### 10. NGFR-Stratified Immune Infiltration

Same immune subpopulations, now broken down by CK⁺ NGFR⁺ vs. CK⁺ NGFR⁻ compartment.

In [ ]:
counts_df_ngfr = plot_combination_counts(
    df=df_binary,
    rois=ROIS_TO_ANALYZE,
    combinations=NGFR_INFILTRATION_COMBINATIONS,
    output_dir=CELL_COUNT_OUTPUT_DIR,
    base_filename="Tumor_Infiltration_NGFR",
    plot_title="NGFR-Related Tumor Infiltration: Counts by Combination and ROI"
)
counts_df_ngfr


### 11. Conditional Cell Overlay

Multi-panel QC figure: composite + per-marker channels + filtered cells for a given condition.
Edit `conditions` to explore other marker combinations.

In [ ]:


conditions = ["CK_mask-", "CD3_intensity+"]

plot_conditional_cells_channels(
    rois=ROIS_TO_ANALYZE,
    conditions=conditions,
    dapi_masks_dict=dapi_masks_dict,
    images_dict=images_dict,
    df_binary=df_binary,
    marker_dict=MARKER_LABELS,          
    ck_masks_dict=ck_masks_dict,
    ngfr_masks_dict=ngfr_masks_dict,
    condition_column_map=CONDITION_COLUMN_MAP,
    brightness_factor=DEFAULT_BRIGHTNESS
)


### 12. Mask Area Summary

Computes CK⁺ area, CK⁺∩NGFR⁺ area, and total ROI area (µm²) per ROI.

In [ ]:
mask_area_summary = compute_mask_area_summary(ck_masks_dict, ngfr_masks_dict)
mask_area_summary


### 13. Cell Densities per Subpopulation

Cells per µm² inside CK⁺ and CK⁺∩NGFR⁺ compartments for each immune subpopulation and ROI.

In [ ]:
for subpop_name, conditions in SUBPOPULATIONS.items():
    df_summary, df_formatted = compute_subpop_cells_per_area(
        df_binary=df_binary,
        subpop_conditions=conditions,
        cond_map=CONDITION_COLUMN_MAP,
        mask_summary=mask_area_summary,
        rois=ROIS_TO_ANALYZE,
        out_dir=CELL_DENSITY_OUTPUT_DIR,
        roi_col=ROI
    )
    print(f"— Processed {subpop_name}")
    display(df_summary.head())
    display(df_formatted.head())

### 14. Signed Distances: Subpopulation → CK / NGFR Boundary

For each cell in a subpopulation, computes the signed Euclidean distance to the nearest CK and NGFR boundary.
Negative = intratumoral; positive = stromal exclusion.

In [ ]:
for subpop_name, subpop_conditions in SUBPOPULATIONS.items():
    for roi in ROIS_TO_ANALYZE:
        compute_and_save(
            roi=roi,
            subpop_name=subpop_name,
            subpop_conditions=subpop_conditions,
            path_save=str(DISTANCES_SUBPOP_DIR),
            dapi_masks=dapi_masks_dict,
            ck_masks=ck_masks_dict,
            ngfr_masks=ngfr_masks_dict,
            df_bin=df_binary,
            col_map=CONDITION_COLUMN_MAP,
            max_cells=None,
        )

In [ ]:
%matplotlib inline

# 1) Load data
subpop_data = load_csv_data(DISTANCES_SUBPOP_DIR)

# 2) By subpopulation
generate_boxplots_nested(
    nested_data=subpop_data,
    positive_col=DISTANCE_CK_POSITIVE,
    negative_col=DISTANCE_CK_NEGATIVE,
    label="CK",
    output_dir=BOXPLOTS_DISTANCES_DIR,
    prefix="mask2subpop"
)

# 3) Inverted (by ROI)
roi_data = invert_dict(subpop_data)
generate_boxplots_nested(
    nested_data=roi_data,
    positive_col=DISTANCE_CK_POSITIVE,
    negative_col=DISTANCE_CK_NEGATIVE,
    label="CK",
    output_dir=BOXPLOTS_DISTANCES_DIR,
    prefix="mask2subpop_byROI"
)

### 15. Inter-Population Distances with Voronoi Tessellation

Nearest-neighbor distances (µm) from each CK⁺ NGFR⁺ (and CK⁺ NGFR⁻) cell to each immune subpopulation,
visualized as Voronoi overlays and violin/box plots.

In [ ]:
shading_dict = {
    mask_name: ({"CK_mask": ck_masks_dict, "NGFR_mask": ngfr_masks_dict}[mask_name], color)
    for mask_name, color in SHADING_COLORS.items()
}
masks_to_shade = list(shading_dict.keys())

_common = dict(
    df_binary=df_binary,
    dapi_masks_dict=dapi_masks_dict,
    condition_column_map=CONDITION_COLUMN_MAP,
    pixel_size=PIXEL_SIZE,
    masks_to_shade=masks_to_shade,
    shading_dict=shading_dict,
    save_matrix_as_csv=True,
    plot_type="voronoi",
)

# A = CK+ NGFR+  vs each immune subpopulation
for subpop_b_label, subpop_b_conditions in SUBPOPULATIONS.items():
    path_save = DISTANCES_POPULATIONS_DIR / subpop_b_label
    path_save.mkdir(parents=True, exist_ok=True)
    compute_and_plot_subpop_distances_for_all_rois(
        rois=ROIS_TO_ANALYZE,
        subpop_conditions_A=SUBPOPULATION_A_POSITIVE,
        subpop_conditions_B=subpop_b_conditions,
        path_save=str(path_save),
        subpop_b_label=subpop_b_label,
        **_common,
    )

# A = CK+ NGFR-  vs each immune subpopulation
for subpop_b_label, subpop_b_conditions in SUBPOPULATIONS.items():
    path_save = DISTANCES_POPULATIONS_DIR / subpop_b_label
    path_save.mkdir(parents=True, exist_ok=True)
    compute_and_plot_subpop_distances_for_all_rois(
        rois=ROIS_TO_ANALYZE,
        subpop_conditions_A=SUBPOPULATION_A_NEGATIVE,
        subpop_conditions_B=subpop_b_conditions,
        path_save=str(path_save),
        subpop_b_label=subpop_b_label,
        **_common,
    )

In [ ]:


# 1) Load the data from the population distance directory
subpop_data = load_csv_data(DISTANCES_POPULATIONS_DIR)

# 2) Generate the boxplots/violin plots for both subpopulation and ROI
generate_combined_boxplots(
    dic_distances=subpop_data,
    save_path=BOXPLOTS_DISTANCES_HEATMAPS_DIR
)
